# UAVid++ — DINOv3 ViT-H+ + UNet training notebook

This notebook reproduces the **official UAVid++ ViT-H+ + UNet training pipeline** using the authors' public code repository.

**Target configuration**
- Backbone: DINOv3 ViT-H+/16
- Head: the repository's `UNetHead`
- Dataset: UAVid++ (11 semantic classes)
- Input tiles: 1088 × 1088
- Batch size: 8
- Optimizer: AdamW
- Learning rate: 3e-5
- Weight decay: 0.001
- Epochs: 40
- Warmup: 5 epochs
- Scheduler: linear warmup + cosine annealing
- Augmentations: the exact augmentations in the official `model_train.py`
- Evaluation: per-class IoU, mIoU, overall accuracy, tiled test inference

**Hardware note:** the authors report training the experiments on **1 × NVIDIA A100 40 GB**. ViT-H+ is a large backbone, so a T4/P100/16 GB GPU is not expected to reproduce this configuration reliably without reducing the batch size or using gradient accumulation/checkpointing.

The notebook intentionally clones and executes the authors' official repository code rather than reimplementing the architecture from scratch.


In [ ]:
# Cell 1 — Check GPU and install dependencies

!nvidia-smi

!pip -q install --upgrade pip
!pip -q install huggingface_hub[cli] albumentations opencv-python-headless tqdm matplotlib numpy pillow pyarrow

# Install the repository's Python requirements after cloning it.


In [ ]:
# Cell 2 — Clone the official UAVid++ code

import os, shutil, subprocess, pathlib

REPO_DIR = "/content/uavidplusplus-code"

if os.path.exists(REPO_DIR):
    print("Repository already exists:", REPO_DIR)
else:
    !git clone https://github.com/vivichiciudean/uavidplusplus-code.git {REPO_DIR}

%cd {REPO_DIR}

print("Repository:", os.getcwd())
!find code data_processing -maxdepth 2 -type f | sort | head -100


In [ ]:
# Cell 3 — Install the exact repository requirements

%cd /content/uavidplusplus-code

!python -m pip install -q -r req.txt

# The official README recommends Python 3.10+ and CUDA-enabled PyTorch.
# Keep the Colab/runtime PyTorch if it is already CUDA-compatible.
import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


## Hugging Face access

The UAVid++ dataset repository is gated. You must accept its dataset terms on Hugging Face before downloading it. DINOv3 ViT-H+ weights may also require Hugging Face/Meta access approval.

Create/use a Hugging Face token with permission to read the gated resources. The next cell asks for it without displaying it.


In [ ]:
# Cell 4 — Authenticate to Hugging Face

from huggingface_hub import login
from getpass import getpass

HF_TOKEN = getpass("Enter your Hugging Face access token: ")
login(token=HF_TOKEN)

print("Hugging Face authentication configured.")


In [ ]:
# Cell 5 — Download UAVid++ from the official Hugging Face dataset

from huggingface_hub import snapshot_download
import os

HF_DATASET = "vivianchiciudean/uavidplusplus"
HF_DATA_DIR = "/content/uavidplusplus_hf"

snapshot_download(
    repo_id=HF_DATASET,
    repo_type="dataset",
    local_dir=HF_DATA_DIR,
    token=HF_TOKEN
)

print("Downloaded dataset repository to:", HF_DATA_DIR)

for root, dirs, files in os.walk(HF_DATA_DIR):
    for f in files:
        print(os.path.relpath(os.path.join(root, f), HF_DATA_DIR))


In [ ]:
# Cell 6 — Extract the UAVid++ archives and locate the official splits

import os, zipfile, shutil, glob

PROJECT = "/content/uavidplusplus-code"
DATA_DIR = os.path.join(PROJECT, "data")
RAW_DIR = os.path.join(PROJECT, "data_raw")

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(RAW_DIR, exist_ok=True)

zip_files = glob.glob(os.path.join(HF_DATA_DIR, "**", "*.zip"), recursive=True)

print("ZIP archives found:")
for z in zip_files:
    print(" -", z)

if not zip_files:
    raise FileNotFoundError(
        "No ZIP archives were found in the downloaded UAVid++ repository. "
        "Check that your Hugging Face account has accepted the gated dataset terms."
    )

for z in zip_files:
    name = os.path.splitext(os.path.basename(z))[0]
    out = os.path.join(RAW_DIR, name)
    os.makedirs(out, exist_ok=True)
    print("\nExtracting:", os.path.basename(z))
    with zipfile.ZipFile(z, "r") as zz:
        zz.extractall(out)

# Search for expected split directories.
candidates = []
for root, dirs, files in os.walk(RAW_DIR):
    for d in dirs:
        if d in {"uavid_train", "uavid_val", "uavid_test"}:
            candidates.append(os.path.join(root, d))

print("\nDetected split directories:")
for c in candidates:
    print(" -", c)

# Copy/move each detected split to the exact names expected by the official preprocessing.
for split in ["uavid_train", "uavid_val", "uavid_test"]:
    matches = [p for p in candidates if os.path.basename(p) == split]
    if not matches:
        raise FileNotFoundError(
            f"Could not locate {split}. Inspect the extracted archive layout above."
        )
    src = matches[0]
    dst = os.path.join(DATA_DIR, split)
    if os.path.exists(dst):
        shutil.rmtree(dst)
    shutil.copytree(src, dst)
    print("Prepared:", dst)


## Flatten the official UAVid sequence structure

The repository's `1_flatten.py` converts the sequence structure into flat `Images/` and `Labels/` directories. The original script contains a single hard-coded validation path, so this notebook generates the same operation for all three official splits without changing its flattening logic.


In [ ]:
# Cell 7 — Run the official flattening logic for train/val/test

import os, glob, cv2

PROJECT = "/content/uavidplusplus-code"
DATA_DIR = os.path.join(PROJECT, "data")

def flatten_uavid_split(split):
    original_format_folder = os.path.join(DATA_DIR, split)
    out_folder = os.path.join(DATA_DIR, "flat_" + split)

    for folder in ["Images", "Labels"]:
        os.makedirs(os.path.join(out_folder, folder), exist_ok=True)

        files = sorted(
            glob.glob(
                os.path.join(
                    original_format_folder,
                    "*/",
                    folder,
                    "*.png"
                )
            )
        )

        if not files:
            raise FileNotFoundError(
                f"No PNG files found for {split}/{folder}. "
                f"Expected sequence directories under {original_format_folder}."
            )

        for image_name in files:
            img = cv2.imread(image_name)
            parts = image_name.replace("\\", "/").split("/")
            seq = int(parts[-3].replace("seq", ""))
            number = int(parts[-1].replace(".png", ""))
            image_id = seq * 1000 + number

            out_name = os.path.join(
                out_folder,
                folder,
                f"{image_id}.png"
            )
            cv2.imwrite(out_name, img)

    print(
        f"{split}: flattened "
        f"{len(glob.glob(os.path.join(out_folder, 'Images', '*.png')))} images"
    )

for split in ["uavid_train", "uavid_val", "uavid_test"]:
    flatten_uavid_split(split)


## Tile the 4K frames exactly as the official pipeline

The official DINOv3 preprocessing uses **1088 × 1088 non-overlapping tiles**, with centered padding and mask padding value 255. Training and validation are tiled; the test frames remain full-resolution because the official training script performs tiled inference over the test images.


In [ ]:
# Cell 8 — Run the official 1088×1088 preprocessing for train and validation

import subprocess, os

PROJECT = "/content/uavidplusplus-code"

def run_preprocess(input_dir, output_img_dir, output_mask_dir):
    cmd = [
        "python",
        os.path.join(PROJECT, "data_processing", "2_preprocess_images.py"),
        "--input-dir", input_dir,
        "--output-img-dir", output_img_dir,
        "--output-mask-dir", output_mask_dir,
        "--mode", "train",
        "--split-size-h", "1088",
        "--split-size-w", "1088",
        "--stride-h", "1088",
        "--stride-w", "1088",
    ]
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, check=True, cwd=os.path.join(PROJECT, "data_processing"))

run_preprocess(
    os.path.join(PROJECT, "data", "flat_uavid_train"),
    os.path.join(PROJECT, "data", "train", "Images"),
    os.path.join(PROJECT, "data", "train", "Labels"),
)

run_preprocess(
    os.path.join(PROJECT, "data", "flat_uavid_val"),
    os.path.join(PROJECT, "data", "val", "Images"),
    os.path.join(PROJECT, "data", "val", "Labels"),
)

print("Train/validation tiling complete.")


In [ ]:
# Cell 9 — Prepare the official test split

# The official dataloader expects:
# data/test/Images/*.png
# data/test/Labels/*.png
#
# Test images are kept at full resolution for tiled inference.

import os, shutil

PROJECT = "/content/uavidplusplus-code"

test_src = os.path.join(PROJECT, "data", "flat_uavid_test")
test_dst = os.path.join(PROJECT, "data", "test")

if os.path.exists(test_dst):
    shutil.rmtree(test_dst)

os.makedirs(test_dst, exist_ok=True)

shutil.copytree(
    os.path.join(test_src, "Images"),
    os.path.join(test_dst, "Images")
)

shutil.copytree(
    os.path.join(test_src, "Labels"),
    os.path.join(test_dst, "Labels")
)

print("Test images:",
      len(os.listdir(os.path.join(test_dst, "Images"))))

print("Test labels:",
      len(os.listdir(os.path.join(test_dst, "Labels"))))


## Download DINOv3 ViT-H+ exactly required by the official code

The official `model_train.py` expects:

`dinov3_vith16plus_pretrain_lvd1689m-7c1da9a5.pth`

and the model name:

`dinov3_vith16plus`

The corresponding official Hugging Face model is `facebook/dinov3-vith16plus-pretrain-lvd1689m`.


In [ ]:
# Cell 10 — Clone DINOv3 and download the exact ViT-H+ checkpoint

import os, subprocess
from huggingface_hub import hf_hub_download

PROJECT = "/content/uavidplusplus-code"
MODELS_DIR = os.path.join(PROJECT, "models")
DINO_REPO_DIR = os.path.join(MODELS_DIR, "dinov3")

os.makedirs(MODELS_DIR, exist_ok=True)

if not os.path.exists(DINO_REPO_DIR):
    !git clone https://github.com/facebookresearch/dinov3.git {DINO_REPO_DIR}

DINO_REPO = "facebook/dinov3-vith16plus-pretrain-lvd1689m"
DINO_FILE = "dinov3_vith16plus_pretrain_lvd1689m-7c1da9a5.pth"

dino_path = hf_hub_download(
    repo_id=DINO_REPO,
    filename=DINO_FILE,
    token=HF_TOKEN,
    local_dir=MODELS_DIR
)

print("DINOv3 ViT-H+ checkpoint:")
print(dino_path)

assert os.path.exists(
    os.path.join(MODELS_DIR, DINO_FILE)
)


## Configure the authors' exact training script for ViT-H+ + UNet

The repository's `model_train.py` defaults to ViT-S for its smoke-test configuration. The notebook below creates a training copy with only the experiment selection changed to the official **ViT-H+ + UNet** configuration:

- H+ checkpoint filename
- `MODEL_NAME = "dinov3_vith16plus"`
- experiment version name

All model, loss, augmentation, optimizer, scheduler, tiling, checkpointing, and evaluation code remains the authors' implementation.


In [ ]:
# Cell 11 — Create the exact ViT-H+ UNet training entry point

from pathlib import Path

PROJECT = Path("/content/uavidplusplus-code")
original = PROJECT / "code" / "model_train.py"
target = PROJECT / "code" / "model_train_vithplus_unet_uavidpp.py"

text = original.read_text()

# Switch the selected DINOv3 backbone from ViT-S to ViT-H+.
text = text.replace(
    'WEIGHTS_NAME = "../models/dinov3_vits16_pretrain_lvd1689m-08c60483.pth"',
    'WEIGHTS_NAME = "../models/dinov3_vith16plus_pretrain_lvd1689m-7c1da9a5.pth"'
)

text = text.replace(
    'MODEL_NAME = "dinov3_vits16"',
    'MODEL_NAME = "dinov3_vith16plus"'
)

text = text.replace(
    'version = "9_dinov3_vits_unet_inv_uavid++"',
    'version = "9_dinov3_vithplus_unet_uavid++"'
)

target.write_text(text)

print("Created:", target)
print("\nSelected configuration:")
for line in text.splitlines():
    if (
        "WEIGHTS_NAME =" in line
        or "MODEL_NAME =" in line
        or "version =" in line
        or "learning_rate =" in line
        or "weight_decay =" in line
        or "epochs =" in line
        or "BATCH_SIZE =" in line
        or "HEIGHT =" in line
        or "WIDTH =" in line
    ):
        print(line)


In [ ]:
# Cell 12 — Sanity-check the official model and data before the long run

%cd /content/uavidplusplus-code/code

# Import the same modules used by model_train.py.
import torch
from dataloader import *
from segmentation_heads import UNetHead

print("num_labels:", num_labels)
print("id2name:", id2name)

# Expected UAVid++ classes in the official code:
# clutter, wall, road, tree, lowveg, water, sky, roof,
# staticcar, dynamiccar, human

assert num_labels == 11, num_labels

print("\nCUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM GB:", torch.cuda.get_device_properties(0).total_memory / 1024**3)

print("\nTrain tiles:",
      len(glob.glob("../data/train/Images/*.png")))
print("Val tiles:",
      len(glob.glob("../data/val/Images/*.png")))
print("Test frames:",
      len(glob.glob("../data/test/Images/*.png")))


# Full training

The next cell launches the **authors' actual training script**. It performs:
1. dataset loading
2. class-weight computation
3. DINOv3 ViT-H+ feature extraction with the backbone frozen
4. UNet-head training
5. AdamW optimization
6. 5-epoch linear warmup
7. cosine annealing
8. validation mIoU/OA each epoch
9. best-model checkpointing
10. final tiled test inference
11. per-class IoU, mIoU and OA

Expect a long runtime on A100. Do not interrupt the cell unless necessary.


In [ ]:
# Cell 13 — RUN COMPLETE ViT-H+ + UNet TRAINING

%cd /content/uavidplusplus-code/code

!python model_train_vithplus_unet_uavidpp.py


In [ ]:
# Cell 14 — Inspect the generated outputs

import os, glob

PROJECT = "/content/uavidplusplus-code"

print("Output files:")
for p in sorted(glob.glob(os.path.join(PROJECT, "output", "*"))):
    print(" -", os.path.relpath(p, PROJECT))

print("\nTraining logs:")
for p in sorted(glob.glob(os.path.join(PROJECT, "output", "training_log_*.txt"))):
    print(" -", p)

print("\nBest checkpoint:")
print(os.path.join(PROJECT, "output", "best_model.pth"))


In [ ]:
# Cell 15 — Display training curves if they were generated

from IPython.display import display, Image as IPImage
import os

PROJECT = "/content/uavidplusplus-code"
output = os.path.join(PROJECT, "output")

for name in [
    "5_9_dinov3_vithplus_unet_uavid++_loss.png",
    "6_9_dinov3_vithplus_unet_uavid++_iou.png",
    "7_9_dinov3_vithplus_unet_uavid++_accuracy.png",
]:
    path = os.path.join(output, name)
    if os.path.exists(path):
        print("\n", name)
        display(IPImage(filename=path))


In [ ]:
# Cell 16 — Display a few test predictions

from IPython.display import display, Image as IPImage
import os, glob

PROJECT = "/content/uavidplusplus-code"
pred_dir = os.path.join(
    PROJECT,
    "output",
    "test_predictions_9_dinov3_vithplus_unet_uavid++"
)

files = sorted(glob.glob(os.path.join(pred_dir, "*.png")))

print("Prediction count:", len(files))

for path in files[:10]:
    print(os.path.basename(path))
    display(IPImage(filename=path))


## Notes

- The official UAVid++ paper/repository reports **81.42% mIoU on UAVid++ for DINOv3 ViT-H+ + UNet** and **82.04% for ViT-7B + UNet** under their reported protocol.
- The repository's official training configuration uses 1088×1088 tiles, batch size 8, 40 epochs, AdamW with lr 3e-5 and weight decay 0.001, 5-epoch warmup followed by cosine annealing, and the augmentation pipeline used in `model_train.py`.
- The notebook does not invent a replacement architecture; it executes the official repository implementation.
- The UAVid++ dataset is gated and licensed CC BY-NC-SA 4.0. The underlying UAVid imagery has its own licensing requirements.
- DINOv3 ViT-H+ is a large 3.36 GB checkpoint; training the full configuration is intended for substantial GPU memory, with the paper's reported setup using an A100 40 GB.
